# IntZ Example 16: SPARC–IntZ Kinematic Bridge

**EPS Research — Flynn, D.C. (2026)**

Direct comparison of observed kinematics between SPARC (z=0) and
KROSS Tier-1 (z~0.6–1.0): circular velocity, velocity dispersion,
and kinematic state V/σ across 7 Gyr of cosmic evolution.

> **Note (August 2026):** This notebook previously compared SPARC and
> IntZ omega values. The IntZ omega values have been withdrawn — see
> [CORRECTIONS.md](../../CORRECTIONS.md). Replaced with observed
> kinematic comparison (Vc, σ, V/σ).


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import csv, json, numpy as np

# ── Load SPARC from HI corpus v7 ──────────────────────────────────────────
sparc_rows = []
with open('../../hi_corpus_v7/rotation_curve_corpus_v7.json') as f:
    sparc = json.load(f)
for g in sparc.get('galaxies', []):
    try:
        flat = g.get('kinematic_summary', {})
        vmax = flat.get('vrot_max_kms') or flat.get('Vmax')
        if not vmax:
            # Try from data points
            data = g.get('data', [])
            if data:
                vmax = max(p.get('Vrot', p.get('Vobs', 0)) for p in data)
        if vmax and float(vmax) > 0:
            sparc_rows.append({'Vc': float(vmax), 'z': 0.0})
    except (TypeError, ValueError):
        pass

print(f'SPARC galaxies with Vmax: {len(sparc_rows)}')

# ── Load KROSS Tier-1 from IntZ ───────────────────────────────────────────
kross_rows = []
with open('intz_corpus_v1b_flat.csv') as f:
    for row in csv.DictReader(f):
        if row['survey'] == 'KROSS' and row['quality_tier'] == '1':
            try:
                vc  = float(row['Vc_kms'])
                z   = float(row['z_spec'])
                sig = float(row['sigma0_kms']) if row['sigma0_kms'] else None
                vos = float(row['v_over_sigma']) if row['v_over_sigma'] else None
                if vc > 0:
                    kross_rows.append({'Vc': vc, 'z': z, 'sigma': sig, 'vos': vos})
            except (ValueError, KeyError):
                pass

print(f'KROSS Tier-1 galaxies: {len(kross_rows)}')


In [ ]:
sparc_vc = [r['Vc'] for r in sparc_rows]
kross_vc = [r['Vc'] for r in kross_rows]
kross_sig = [r['sigma'] for r in kross_rows if r['sigma']]
kross_vos = [r['vos'] for r in kross_rows if r['vos'] and 0 < r['vos'] < 25]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Panel 1: Vc comparison
ax = axes[0]
bins = np.linspace(0, 400, 30)
ax.hist(sparc_vc, bins=bins, alpha=0.6, color='#2ca02c',
        label=f'SPARC z=0 (N={len(sparc_vc)}, med={np.median(sparc_vc):.0f})')
ax.hist(kross_vc, bins=bins, alpha=0.6, color='#ff7f0e',
        label=f'KROSS z~0.9 (N={len(kross_vc)}, med={np.median(kross_vc):.0f})')
ax.axvline(np.median(sparc_vc), color='#2ca02c', lw=2, ls='--')
ax.axvline(np.median(kross_vc), color='#ff7f0e', lw=2, ls='--')
ax.set_xlabel('Circular Velocity Vc (km/s)', fontsize=12)
ax.set_ylabel('N galaxies', fontsize=12)
ax.set_title('Vc Distribution\nSPARC vs KROSS', fontsize=11)
ax.legend(fontsize=8)

# Panel 2: Vc CDFs
ax2 = axes[1]
for vc_arr, label, color in [
    (sparc_vc, 'SPARC z=0', '#2ca02c'),
    (kross_vc, 'KROSS z~0.9', '#ff7f0e')
]:
    sorted_vc = np.sort(vc_arr)
    cdf = np.arange(1, len(sorted_vc)+1) / len(sorted_vc)
    ax2.plot(sorted_vc, cdf, color=color, lw=2, label=label)
ax2.set_xlabel('Vc (km/s)', fontsize=12)
ax2.set_ylabel('Cumulative fraction', fontsize=12)
ax2.set_title('Vc Cumulative Distribution', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# Panel 3: σ and V/σ for KROSS
ax3 = axes[2]
ax3.hist(kross_sig, bins=20, alpha=0.7, color='#1f77b4',
         label=f'σ₀ (med={np.median(kross_sig):.1f} km/s)')
ax3_twin = ax3.twinx()
ax3_twin.hist(kross_vos, bins=20, alpha=0.4, color='#d62728',
              label=f'V/σ (med={np.median(kross_vos):.1f})')
ax3.set_xlabel('km/s  /  V/σ ratio', fontsize=11)
ax3.set_ylabel('N (σ₀)', fontsize=11, color='#1f77b4')
ax3_twin.set_ylabel('N (V/σ)', fontsize=11, color='#d62728')
ax3.set_title('KROSS Dispersion + Kinematic State', fontsize=11)
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
ax3.legend(lines1+lines2, labels1+labels2, fontsize=8)

plt.suptitle('SPARC (z=0) — KROSS (z~0.9) Kinematic Bridge\n'
             'Observed Vc, σ₀, V/σ (no omega correction)', fontsize=11)
plt.tight_layout()
plt.savefig('intz_nb16_sparc_intz_kinematic_bridge.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'SPARC median Vc:  {np.median(sparc_vc):.1f} km/s')
print(f'KROSS median Vc:  {np.median(kross_vc):.1f} km/s')
print(f'KROSS median σ₀:  {np.median(kross_sig):.1f} km/s')
print(f'KROSS median V/σ: {np.median(kross_vos):.2f}')
print('Saved: intz_nb16_sparc_intz_kinematic_bridge.png')


## Summary

| Quantity | SPARC z=0 | KROSS z~0.9 |
|----------|-----------|-------------|
| N galaxies | ~175 | 166 |
| Tracer | HI 21cm | Hα (beam-smear corrected) |
| Median Vc | see output | see output |

The SPARC and KROSS samples probe different physical radii and use
different kinematic tracers — direct comparison requires tracer
corrections beyond the scope of this notebook.

**Citation:**
```
Flynn, D.C. (2026). IntZ Kinematic Corpus v1.0.
Zenodo. DOI: 10.5281/zenodo.21841382

Flynn, D.C. & Cannaliato, J. (2025). Frontiers in Astronomy and Space Sciences, 12.
DOI: 10.3389/fspas.2025.1680387
```
